# Task 4.1 — Anti-Spoofing Classifier

In [1]:
!pip install -q soundfile torchaudio matplotlib numpy

In [2]:
import gc, json, math
from pathlib import Path
from typing import List, Tuple
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torchaudio
from torch.utils.data import DataLoader, TensorDataset

TARGET_SR = 16_000
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## Audio loading & LFCC extraction

In [ ]:
def load_mono_16k(path):
    wav_np, sr = sf.read(str(path), always_2d=True, dtype='float32')
    wav = torch.from_numpy(wav_np.T)
    if wav.size(0) > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    return wav.squeeze(0).contiguous()

def extract_lfcc(waveform, n_lfcc=20, n_fft=512, hop_length=160, n_filter=40):
    """Extract LFCC (Linear Frequency Cepstral Coefficients)."""
    window = torch.hann_window(n_fft)
    stft = torch.stft(waveform, n_fft=n_fft, hop_length=hop_length,
                      win_length=n_fft, window=window, return_complex=True)
    power_spec = stft.abs().pow(2)
    n_freqs = power_spec.size(0)

    # Linear filterbank
    filterbank = torch.zeros(n_filter, n_freqs)
    freq_points = torch.linspace(0, n_freqs-1, n_filter+2).long()
    for i in range(n_filter):
        left, center, right = freq_points[i], freq_points[i+1], freq_points[i+2]
        for j in range(left, center):
            if center > left: filterbank[i, j] = (j - left) / (center - left)
        for j in range(center, right):
            if right > center: filterbank[i, j] = (right - j) / (right - center)

    filtered = torch.matmul(filterbank, power_spec)
    log_filtered = torch.log(filtered + 1e-12)

    # DCT
    dct_matrix = torch.zeros(n_lfcc, n_filter)
    for k in range(n_lfcc):
        for n in range(n_filter):
            dct_matrix[k, n] = np.cos(np.pi * k * (2*n+1) / (2*n_filter))

    return torch.matmul(dct_matrix, log_filtered)

def segment_features(waveform, segment_s=2.0, n_lfcc=20):
    """Extract LFCC features per segment with augmentation."""
    seg_samples = int(segment_s * TARGET_SR)
    features = []

    for start in range(0, waveform.numel() - seg_samples + 1, seg_samples):
        segment = waveform[start:start + seg_samples]
        lfcc = extract_lfcc(segment, n_lfcc=n_lfcc)
        delta = torch.zeros_like(lfcc)
        delta[:, 1:] = lfcc[:, 1:] - lfcc[:, :-1]
        delta2 = torch.zeros_like(delta)
        delta2[:, 1:] = delta[:, 1:] - delta[:, :-1]

        feat = torch.cat([
            lfcc.mean(dim=1), lfcc.std(dim=1),
            delta.mean(dim=1), delta.std(dim=1),
            delta2.mean(dim=1), delta2.std(dim=1),
        ])
        features.append(feat)

    # Data augmentation: half-overlapping segments
    half_seg = seg_samples // 2
    for start in range(half_seg, waveform.numel() - seg_samples + 1, seg_samples):
        segment = waveform[start:start + seg_samples]
        lfcc = extract_lfcc(segment, n_lfcc=n_lfcc)
        delta = torch.zeros_like(lfcc)
        delta[:, 1:] = lfcc[:, 1:] - lfcc[:, :-1]
        delta2 = torch.zeros_like(delta)
        delta2[:, 1:] = delta[:, 1:] - delta[:, :-1]
        feat = torch.cat([
            lfcc.mean(dim=1), lfcc.std(dim=1),
            delta.mean(dim=1), delta.std(dim=1),
            delta2.mean(dim=1), delta2.std(dim=1),
        ])
        features.append(feat)

    return torch.stack(features) if features else torch.empty(0)

## Load audio files & extract features

In [4]:
bonafide_files = ['Student_voice_ref.wav', 'original_segment.wav']
spoof_files = ['output_LRL_cloned.wav']

bonafide_feats = []
for f in bonafide_files:
    if Path(f).exists():
        wav = load_mono_16k(f)
        feats = segment_features(wav)
        bonafide_feats.append(feats)
        print(f'  Bona fide: {f} → {feats.size(0)} segments')
    else:
        print(f'  Warning: {f} not found')

spoof_feats = []
for f in spoof_files:
    if Path(f).exists():
        wav = load_mono_16k(f)
        feats = segment_features(wav)
        spoof_feats.append(feats)
        print(f'  Spoof: {f} → {feats.size(0)} segments')
    else:
        print(f'  Warning: {f} not found')

assert bonafide_feats and spoof_feats, 'Need both bona fide and spoof files!'
X_bf = torch.cat(bonafide_feats)
X_sp = torch.cat(spoof_feats)
print(f'\nTotal: {X_bf.size(0)} bona fide, {X_sp.size(0)} spoof')

  Bona fide: Student_voice_ref.wav → 61 segments
  Bona fide: original_segment.wav → 598 segments
  Spoof: output_LRL_cloned.wav → 539 segments

Total: 659 bona fide, 539 spoof


## Deeper classifier with batch normalisation

In [5]:
class AntiSpoofClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

## Train & evaluate

In [6]:
# Prepare dataset
X = torch.cat([X_bf, X_sp])
y = torch.cat([torch.ones(X_bf.size(0)), torch.zeros(X_sp.size(0))])
perm = torch.randperm(X.size(0))
X, y = X[perm], y[perm]

n_train = int(0.8 * X.size(0))
X_train, X_test = X[:n_train].to(device), X[n_train:].to(device)
y_train, y_test = y[:n_train].to(device), y[n_train:].to(device)
print(f'Train: {X_train.size(0)}, Test: {X_test.size(0)}')

# Train
model = AntiSpoofClassifier(input_dim=X.size(1)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
criterion = nn.BCEWithLogitsLoss()

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)

for epoch in range(1, 101):
    model.train()
    total_loss, total_n = 0.0, 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * yb.size(0)
        total_n += yb.size(0)
    scheduler.step()

    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            test_logits = model(X_test)
            test_preds = (torch.sigmoid(test_logits) >= 0.5).float()
            acc = (test_preds == y_test).float().mean().item()
        print(f'  Epoch {epoch:03d} | loss={total_loss/total_n:.4f} | test_acc={acc:.4f}')

Train: 958, Test: 240
  Epoch 010 | loss=0.0172 | test_acc=1.0000
  Epoch 020 | loss=0.0056 | test_acc=1.0000
  Epoch 030 | loss=0.0030 | test_acc=1.0000
  Epoch 040 | loss=0.0026 | test_acc=1.0000
  Epoch 050 | loss=0.0014 | test_acc=1.0000
  Epoch 060 | loss=0.0013 | test_acc=1.0000
  Epoch 070 | loss=0.0011 | test_acc=1.0000
  Epoch 080 | loss=0.0011 | test_acc=1.0000
  Epoch 090 | loss=0.0013 | test_acc=1.0000
  Epoch 100 | loss=0.0010 | test_acc=1.0000


## Compute EER & DET curve

In [7]:
model.eval()
with torch.no_grad():
    test_scores = torch.sigmoid(model(X_test)).cpu().numpy()

test_labels = y_test.cpu().numpy()
bf_scores = test_scores[test_labels == 1]
sp_scores = test_scores[test_labels == 0]

def compute_eer(bf, sp):
    thresholds = np.sort(np.concatenate([bf, sp]))
    best_eer, best_thresh = 1.0, 0.5
    for t in thresholds:
        far = np.mean(sp >= t)
        frr = np.mean(bf < t)
        if far <= frr:
            best_eer = (far + frr) / 2
            best_thresh = t
            break
    return best_eer, best_thresh

eer, threshold = compute_eer(bf_scores, sp_scores)
print(f'EER: {eer*100:.2f}%')
print(f'Threshold: {threshold:.4f}')
print(f'Target: EER < 10%  →  {"PASS ✓" if eer < 0.10 else "FAIL ✗"}')

# DET curve
thresholds = np.linspace(min(bf_scores.min(), sp_scores.min()),
                         max(bf_scores.max(), sp_scores.max()), 200)
fars = [np.mean(sp_scores >= t)*100 for t in thresholds]
frrs = [np.mean(bf_scores < t)*100 for t in thresholds]

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fars, frrs, linewidth=2, label=f'EER = {eer*100:.2f}%')
ax.plot([0,100], [0,100], '--', alpha=0.3, color='gray')
ax.set_xlabel('False Acceptance Rate (%)')
ax.set_ylabel('False Rejection Rate (%)')
ax.set_title('Anti-Spoofing DET Curve')
ax.legend(); ax.grid(alpha=0.2); ax.set_xlim(0,50); ax.set_ylim(0,50)
fig.tight_layout()
fig.savefig('antispoof_det.png', dpi=200)
plt.show()

EER: 0.00%
Threshold: 0.9951
Target: EER < 10%  →  PASS ✓


## Save results

In [8]:
results = {
    'eer_percent': round(eer*100, 2),
    'threshold': round(float(threshold), 4),
    'n_bonafide_test': int(len(bf_scores)),
    'n_spoof_test': int(len(sp_scores)),
    'pass': bool(eer < 0.10),
}
with open('antispoof_results.json', 'w') as f:
    json.dump(results, f, indent=2)

torch.save(model.state_dict(), 'antispoof_model.pt')
print(f'Saved results & model')
print('Done!')

Saved results & model
Done!
